# UD5 - Visualización interactiva con Plotly
## De gráficos estáticos a gráficos que responden

En la sesión anterior creamos gráficos con Matplotlib y Seaborn. Funcionan bien para informes y notebooks, pero tienen una limitación: **son imágenes estáticas**. No puedes pasar el ratón por encima para ver un valor exacto, ni hacer zoom en una zona concreta, ni filtrar datos sobre la marcha.

**Plotly** resuelve eso. Genera gráficos interactivos: con hover (tooltip al pasar el ratón), zoom, pan, y la posibilidad de exportar a HTML para compartir sin necesidad de Python.

Vamos a usar los **mismos datasets de StreamFlow** que ya conocéis, para que veáis la diferencia entre hacer un gráfico con Matplotlib/Seaborn y hacerlo con Plotly.

### Plotly Express vs Plotly Graph Objects

Plotly tiene dos niveles:
- **Plotly Express (px)**: la versión rápida. Una línea de código = un gráfico. Equivalente a lo que es Seaborn para Matplotlib.
- **Plotly Graph Objects (go)**: control total. Más código, pero puedes personalizar absolutamente todo.

Nosotros trabajaremos casi siempre con **Plotly Express**, que cubre el 90% de lo que necesitamos.

## 0. Instalación y configuración

In [1]:
!pip install plotly
!pip install statsmodels

import pandas as pd
import numpy as np
import plotly.express as px          # La API rápida (lo que más usaremos)
import plotly.graph_objects as go    # Para cuando necesitemos control total
import plotly.io as pio
pio.renderers.default = "notebook"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 13.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.0/447.0 kB 14.7 MB/s eta 0:00:00


## 1. Los mismos datos de StreamFlow

Cargamos exactamente los mismos datasets que usamos con Matplotlib y Seaborn. Así la comparación es directa.

In [2]:
# =============================================
# DATASET 1: Datos mensuales de la plataforma
# =============================================

datos_mensuales = pd.DataFrame({
    'mes': ['Ene', 'Feb', 'Mar', 'Abr', 'May', 'Jun', 
            'Jul', 'Ago', 'Sep', 'Oct', 'Nov', 'Dic'],
    'mes_num': range(1, 13),
    'mau': [1200000, 1250000, 1310000, 1350000, 1380000, 1340000,
            1290000, 1260000, 1350000, 1420000, 1500000, 1580000],
    'ingresos_k': [320, 335, 352, 360, 370, 355,
                   338, 325, 358, 385, 410, 468],
    'horas_escucha': [1.8, 1.9, 2.0, 1.9, 1.8, 1.7,
                      1.6, 1.5, 1.8, 2.0, 2.1, 2.3],
    'churn_pct': [3.2, 3.0, 2.8, 2.9, 3.1, 3.5,
                  3.8, 4.0, 3.3, 2.7, 2.5, 2.3],
    'conversion_pct': [4.5, 4.7, 5.0, 4.8, 4.6, 4.3,
                       4.1, 3.9, 4.5, 5.1, 5.5, 6.2],
})

datos_mensuales.head()

,mes,mes_num,mau,ingresos_k,horas_escucha,churn_pct,conversion_pct
0,Ene,1,1200000,320,1.8,3.2,4.5
1,Feb,2,1250000,335,1.9,3.0,4.7
2,Mar,3,1310000,352,2.0,2.8,5.0
3,Abr,4,1350000,360,1.9,2.9,4.8
4,May,5,1380000,370,1.8,3.1,4.6


In [3]:
# =============================================
# DATASET 2: Usuarios individuales
# =============================================

np.random.seed(42)
n_usuarios = 500

usuarios = pd.DataFrame({
    'plan': np.random.choice(['Free', 'Premium', 'Familiar', 'Estudiante'], 
                              n_usuarios, 
                              p=[0.40, 0.30, 0.15, 0.15]),
    'edad': np.random.randint(16, 65, n_usuarios),
    'horas_mes': np.clip(
        np.random.normal(45, 20, n_usuarios),
        5, 150
    ).round(1),
    'pais': np.random.choice(
        ['España', 'México', 'Argentina', 'Colombia', 'Chile', 'Perú'],
        n_usuarios,
        p=[0.30, 0.25, 0.15, 0.13, 0.10, 0.07]
    ),
    'playlists_creadas': np.random.poisson(5, n_usuarios),
})

usuarios.loc[usuarios['plan'] == 'Premium', 'horas_mes'] *= 1.6
usuarios.loc[usuarios['plan'] == 'Free', 'horas_mes'] *= 0.7
usuarios['horas_mes'] = usuarios['horas_mes'].round(1)

usuarios['gasto_mensual'] = usuarios['plan'].map({
    'Free': 0, 'Premium': 9.99, 'Familiar': 14.99, 'Estudiante': 4.99
})

usuarios.head()

,plan,edad,horas_mes,pais,playlists_creadas,gasto_mensual
0,Free,32,13.4,Colombia,6,0.00
1,Estudiante,24,19.1,Colombia,2,4.99
2,Familiar,48,38.3,Colombia,7,14.99
3,Premium,35,125.4,México,4,9.99
4,Free,28,27.9,Colombia,4,0.00


---

## 2. Gráfico de líneas — Evolución temporal

**Pregunta que responde**: ¿Cómo han evolucionado los ingresos de StreamFlow a lo largo del año?

Con Matplotlib hacíamos `plt.plot()`. Con Plotly Express es `px.line()`.

La diferencia: **pasa el ratón por encima de cualquier punto**.

In [23]:
fig = px.line(
    datos_mensuales,           
    x='mes',                   
    y='ingresos_k',            
    title='Evolución de ingresos mensuales (StreamFlow)',
    labels={'ingresos_k': 'Ingresos (miles €)', 'mes': 'Mes'},  
    markers=True,              
)

fig.show()

### Fíjate en lo que puedes hacer:
- **Hover**: al pasar el ratón ves el valor exacto de cada punto.
- **Zoom**: selecciona un área arrastrando para hacer zoom.
- **Pan**: mueve el gráfico con el botón de desplazar.
- **Barra de herramientas**: arriba a la derecha tienes opciones para exportar como PNG, resetear zoom, etc.

Todo eso con **una sola línea** de código (más parámetros de estilo).

### 2.1 Varias líneas: churn vs conversión

¿Los meses con menos abandono coinciden con más conversión? Vamos a superponer ambas métricas.
Para meter varias columnas Y en Plotly Express, le pasamos una lista al parámetro y.

In [5]:
fig = px.line(
    datos_mensuales,
    x='mes',
    y=['churn_pct', 'conversion_pct'],
    title='Churn vs Conversión',
    markers=True,
    labels={'value': '%', 'variable': 'Métrica'},
)
fig.show()

---

## 3. Gráfico de barras — Comparar cantidades

**Pregunta**: ¿De qué país son nuestros usuarios?

In [6]:
# Primero contamos (igual que hacíamos con Matplotlib)
usuarios_por_pais = usuarios['pais'].value_counts().reset_index() #convierte esa Serie en un DataFrame normal con dos columnas
usuarios_por_pais.columns = ['pais', 'cantidad']
fig = px.bar(
    usuarios_por_pais,
    x='pais',
    y='cantidad',
    
    title='Usuarios por país',
    labels={'cantidad': 'Nº de usuarios', 'pais': 'País'},
    color='pais',                # Un color por país
    text='cantidad',             # Valor encima de cada barra
)

fig.show()

### 3.1 Barras agrupadas por plan y país

**Pregunta**: ¿Cómo se distribuyen los planes en cada país?

In [7]:
# Contamos usuarios por país y plan
por_pais_plan = usuarios.groupby(['pais', 'plan']).size().reset_index(name='cantidad')


fig = px.bar(
    por_pais_plan,
    x='pais',
    y='cantidad',
    color='plan',                    # Una barra por plan dentro de cada país
    barmode='group',                 # 'group' = agrupadas, 'stack' = apiladas
    title='Distribución de planes por país',
    labels={'cantidad': 'Usuarios', 'pais': 'País'},
)

fig.show()
fig.write_html('grafico.html')

### 3.2 Apiladas

Cambia `barmode='group'` por `barmode='stack'` en la celda anterior.

Apiladas es mejor cuando quieres ver el total por país. Agrupadas es mejor cuando quieres comparar planes entre sí.

In [8]:
# Contamos usuarios por país y plan
por_pais_plan = usuarios.groupby(['pais', 'plan']).size().reset_index(name='cantidad')
#por_pais_plan = usuarios.groupby(['pais', 'plan'])['edad'].count().reset_index(name='cantidad') #Es lo mismo


fig = px.bar(
    por_pais_plan,
    x='pais',
    y='cantidad',
    color='plan',                    # Una barra por plan dentro de cada país
    barmode='stack',                 # 'group' = agrupadas, 'stack' = apiladas
    title='Distribución de planes por país',
    labels={'cantidad': 'Usuarios', 'pais': 'País'},
)

fig.show()
fig.write_html('grafico.html')

---

## 4. Gráfico de tarta y treemap — Proporciones

**Pregunta**: ¿Qué porcentaje de usuarios tiene cada plan?

In [9]:
plan_counts = usuarios['plan'].value_counts().reset_index()
plan_counts.columns = ['plan', 'cantidad']

fig = px.pie(
    plan_counts,
    values='cantidad',
    names='plan',
    title='Distribución de usuarios por plan',
    hole=0.4,                   # Convierte la tarta en un donut
)

fig.show()

### 4.1 Treemap — Cuando hay muchas categorías

Recordad: la tarta funciona con 2-5 categorías. Si tenemos más, el treemap es mejor.

Vamos a cruzar **país y plan** en un treemap jerárquico.

In [10]:
fig = px.treemap(
    usuarios,
    path=['pais', 'plan'],       # Jerarquía: primero país, dentro plan
    title='Usuarios por país y plan (treemap)',
)

fig.show()

Haz clic en un país para hacer drill-down (entrar en el desglose por plan dentro de ese país). Haz clic en el encabezado superior para volver atrás.

---

## 5. Histograma — Distribución

**Pregunta**: ¿Cómo se distribuyen las horas de escucha mensuales?

In [24]:
fig = px.histogram(
    usuarios,
    x='horas_mes',
    nbins=20,
    title='Distribución de horas de escucha mensuales',
    labels={'horas_mes': 'Horas/mes'},
)

fig.show()

### 5.1 Histograma separado por plan

¿Los usuarios Premium escuchan más que los Free? Veámoslo superponiendo las distribuciones.

In [12]:
fig = px.histogram(
    usuarios,
    x='horas_mes',
    color='plan',                # Un histograma por plan, superpuestos
    nbins=20,
    barmode='overlay',           # 'overlay' = superpuestos, 'stack' = apilados
    opacity=0.6,                 # Transparencia para que se vean las capas
    title='Horas de escucha por plan',
    labels={'horas_mes': 'Horas/mes'},
)

fig.show()

Puedes hacer clic en la leyenda para ocultar/mostrar planes individualmente. Prueba a dejar solo Premium y Free para ver la diferencia clara.

---

## 6. Scatter — Relaciones entre variables

**Pregunta**: ¿Los usuarios que crean más playlists escuchan más horas?

In [13]:
fig = px.scatter(
    usuarios,
    x='playlists_creadas',
    y='horas_mes',
    color='plan',                # Color por plan
    size='gasto_mensual',        # Tamaño del punto según gasto (bubble chart)
    hover_data=['pais', 'edad'], # Info extra al pasar el ratón
    title='Playlists creadas vs Horas de escucha',
    labels={
        'playlists_creadas': 'Playlists creadas',
        'horas_mes': 'Horas/mes',
    },
    opacity=0.7,
)

fig.show()

Fíjate en `hover_data`: al pasar el ratón por un punto ves el país y la edad de ese usuario, sin que esas variables estén representadas en los ejes. Esto es algo que con gráficos estáticos no puedes hacer.

Y `size='gasto_mensual'` convierte el scatter en un **bubble chart**: los puntos Free son pequeños (gasto 0) y los Familiar son los más grandes.

### 6.1 Añadir línea de tendencia

Plotly puede calcular y dibujar la regresión directamente.

In [14]:
fig = px.scatter(
    usuarios,
    x='playlists_creadas',
    y='horas_mes',
    color='plan',
    trendline='ols',             # Ordinary Least Squares = regresión lineal
    title='Playlists vs Horas con línea de tendencia por plan',
    labels={
        'playlists_creadas': 'Playlists creadas',
        'horas_mes': 'Horas/mes',
    },
    opacity=0.6,
)

fig.show()

Si pasas el ratón por la línea de tendencia, ves la ecuación de la regresión y el R². Plotly lo calcula solo.

---

## 7. Box plot — Distribución por categoría

In [15]:
fig = px.box(
    usuarios,
    x='plan',
    y='horas_mes',
    color='plan',
    title='Distribución de horas de escucha por plan',
    labels={'horas_mes': 'Horas/mes'}
)

fig.show()

Pasa el ratón por la caja: ves mediana, Q1, Q3, bigotes y outliers con sus valores exactos. En Matplotlib te los tenías que imaginar.

---

## 8. Personalización y estilo

### Plantillas predefinidas

Plotly incluye temas que aplican un estilo completo con una sola línea.

In [16]:
# Prueba distintos templates:
# 'plotly', 'plotly_white', 'plotly_dark', 'ggplot2', 'seaborn', 'simple_white'

fig = px.line(
    datos_mensuales,
    x='mes',
    y='ingresos_k',
    title='Ingresos mensuales (tema oscuro)',
    markers=True,
    template='plotly_dark',       # Cambia este valor y vuelve a ejecutar
)

fig.show()

---

## 9. Exportar gráficos

### Como HTML interactivo

La gran ventaja de Plotly: puedes guardar un gráfico como archivo `.html` que cualquier persona puede abrir en el navegador **sin necesidad de Python**.

In [17]:
fig = px.bar(
    usuarios_por_pais,
    x='pais',
    y='cantidad',
    title='Usuarios por país',
    text='cantidad',
)

# Guardar como HTML (interactivo)
fig.write_html('usuarios_por_pais.html')
print('Guardado como usuarios_por_pais.html')


Guardado como usuarios_por_pais.html


---

## 10. Mini-dashboard con subplots

Igual que hicimos con Matplotlib (`fig, axs = plt.subplots()`), en Plotly podemos combinar varios gráficos en una sola figura. Esto es un mini-dashboard dentro del notebook.

Vamos a crear un panel con 4 métricas clave de StreamFlow.

In [18]:
from plotly.subplots import make_subplots

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        'Ingresos mensuales',
        'MAU (usuarios activos)',
        'Tasa de churn',
        'Tasa de conversión'
    )
)
#Vamos añadiendo una a una las gráicas al gráfico completo
# Gráfico 1: Ingresos
fig.add_trace(
    go.Scatter(x=datos_mensuales['mes'], y=datos_mensuales['ingresos_k'],
               mode='lines+markers', name='Ingresos (k€)'),
    row=1, col=1
)

# Gráfico 2: MAU
fig.add_trace(
    go.Bar(x=datos_mensuales['mes'], y=datos_mensuales['mau'],
           name='MAU'),
    row=1, col=2
)

# Gráfico 3: Churn
fig.add_trace(
    go.Scatter(x=datos_mensuales['mes'], y=datos_mensuales['churn_pct'],
               mode='lines+markers', name='Churn (%)',
               line=dict(color='red')),
    row=2, col=1
)

# Gráfico 4: Conversión
fig.add_trace(
    go.Scatter(x=datos_mensuales['mes'], y=datos_mensuales['conversion_pct'],
               mode='lines+markers', name='Conversión (%)',
               line=dict(color='green')),
    row=2, col=2
)

fig.update_layout(
    height=600,
    width=1000,
    title_text='Panel de KPIs — StreamFlow 2025',
    showlegend=False,
    template='plotly_white',
)

fig.show()

---

## 11. Ejercicio clase

Vamos a poner en práctica todo lo que hemos visto. Usando el dataset de `usuarios`, cread los siguientes gráficos con Plotly:

**Ejercicio A**: Un gráfico de barras horizontales (`px.bar` con `orientation='h'`) que muestre la **media de horas de escucha por país**, ordenado de mayor a menor.

> Pista: `usuarios.groupby('pais')['horas_mes'].mean()` os da la media. Luego `.sort_values()` para ordenar y `.reset_index()` para convertirlo en DataFrame.

**Ejercicio B**: Un scatter plot que muestre **edad vs horas de escucha**, coloreado por plan, con `hover_data` que incluya el país y las playlists creadas.

**Ejercicio C**: Un gráfico de tarta (donut) que muestre la distribución de usuarios **por país**. ¿Funciona bien con 6 categorías o sería mejor otro tipo de gráfico? Justificad.

**Ejercicio D**: Exportad uno de vuestros gráficos como `.html`, descargadlo y abridlo en el navegador.

In [19]:
# Ejercicio A: Barras horizontales - media de horas por país
# Tu código aquí


In [20]:
# Ejercicio B: Scatter edad vs horas
# Tu código aquí


In [21]:
# Ejercicio C: Tarta / donut por país
# Tu código aquí


In [22]:
# Ejercicio D: Exportar como HTML
# Tu código aquí
